# Track 1 · Classifier export to ONNX for integration (deliver to Rayan before 7 Nov 2026)

AI251P Capstone, Track 1 task (b). Owner: Abdulmohsen Alharthi.

- Exports the selected classifier (`cls-full-64-s43-lr0.002`) to ONNX, checks it against PyTorch on the **validation** crops and measures latency.
- **Never** reads the test crops, test rows or Al-Hada: every file opened under the Drive folder is checked (a path containing `Test` or `AlHada` stops the notebook). No TestLog entry (validation only).
- **Raw logits only.** The checkpoint's `logit_bias` is never applied.
- No W&B, no secrets. One Drive root: `DRIVE_PROJECT`. Every path written into an output is relative to it.
- A missing input stops the notebook; nothing is guessed.
- Output: `02_Classification/Export/<YYYY-MM-DD>_ClsResNet18_s43_vN/`, written as `..._INCOMPLETE` and renamed only when every check passes (never overwritten).
- Code copied verbatim: `class Pretrained`, `MEAN`/`STD` and `gpu_eval` from `notebooks/track1/train_cls_2026-10-25.ipynb`; `square()` from `notebooks/track1/alhada_eval_v2_cell_2026-10-26.py`.

Runtime: Colab, **T4 GPU**.

## 0. Settings, packages, mount Drive

In [ ]:
!pip -q install onnx onnxruntime-gpu      # versions are recorded in export_report.json

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'   # the only Drive path

# Selected model, fixed before export: ResNet18 full fine-tune, lr 0.002, 64 px; among seeds 42/43/44 the highest RAW
# val macro-F1 (W&B best/val/macro_f1: s43 0.8228, s42 0.8165, s44 0.8113). Test and Al-Hada were NOT used to choose.
SELECTION_RULE = ('ResNet18 full fine-tune, lr 0.002, 64 px; among seeds 42/43/44 the highest raw val macro-F1 '
                  '(W&B best/val/macro_f1: s43 0.8228, s42 0.8165, s44 0.8113). Test and Al-Hada were not used to choose.')
IN = {   # relative to DRIVE_PROJECT, read-only
    'ckpt':       '02_Classification/Checkpoints/2026-10-23_cls-full-64-s43-lr0.002_v4.pt',
    'val_npy':    '01_Data/Processed/Crops/2026-10-20_CropsVal_v1.npy',
    'crop_index': '01_Data/Processed/Crops/2026-10-20_CropIndex_v1.csv.gz',
}
CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']
EXPECT = {'img_size': 64, 'val_crops': 13_800, 'crop_side': 96,
          'val_macro_f1': 0.822755, 'val_macro_f1_tol': 0.002,      # W&B best/val/macro_f1 of cls-full-64-s43-lr0.002
          'val_acc_wandb': 0.9453}                                  # info only
OPSET, PARITY_ATOL, BATCH = 17, 1e-3, 512
LAT_WARMUP, LAT_RUNS, LAT_BATCHES = 50, 500, [1, 32]
E2E_BOXES, E2E_FRAME_HW, E2E_SEED, E2E_SIDE_PX = [1, 20, 100], (1080, 1920), 42, (15, 80)   # synthetic frame, see section 7
FORBIDDEN = ('Test', 'AlHada')                 # no file whose path contains these is opened under the Drive folder
assert not [v for v in IN.values() if any(f in v for f in FORBIDDEN)], 'an input path points at test or Al-Hada'

from google.colab import drive
drive.mount('/content/drive')

## 1. Guard, inputs, check log

From here on every `open()` of a file under the Drive folder (this includes `np.load`, `pd.read_csv`, `torch.load`, `onnx.load`) goes through a guard that stops the notebook if the path contains `Test` or `AlHada`.

In [ ]:
import os, io, re, json, time, hashlib, builtins, inspect, importlib.util
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
from PIL import Image
from sklearn.metrics import f1_score
import onnx, onnxruntime as ort

ROOT = Path(DRIVE_PROJECT)
assert ROOT.is_dir(), f'DRIVE_PROJECT not found: {DRIVE_PROJECT}'
_builtin_open = getattr(builtins.open, '_unguarded', builtins.open)
def _guarded_open(file, *args, **kwargs):
    if isinstance(file, (str, bytes, os.PathLike)):
        s = os.fsdecode(file)
        if s.startswith(('/content/drive', DRIVE_PROJECT)) and any(f in s for f in FORBIDDEN):
            raise PermissionError(f'forbidden path (test / Al-Hada) opened: {s}')
    return _builtin_open(file, *args, **kwargs)
_guarded_open._unguarded = _builtin_open
builtins.open = io.open = _guarded_open          # io.open too: zipfile opens the Raw zips through io.open

P = {k: ROOT / v for k, v in IN.items()}
missing = [f'{k}: {v}' for k, v in IN.items() if not P[k].is_file()]
if missing:
    print('MISSING INPUTS (nothing was written):'); print('\n'.join('  ' + m for m in missing))
    raise FileNotFoundError(f'{len(missing)} missing input(s), see the list above')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False   # plain fp32 everywhere
torch.backends.cudnn.benchmark = False
VERSIONS = {'python': os.sys.version.split()[0], 'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'onnx': onnx.__version__, 'onnxruntime': ort.__version__, 'cuda (torch)': torch.version.cuda,
            'cudnn': torch.backends.cudnn.version(), 'pillow': Image.__version__, 'gpu': GPU, 'ort_available_providers': ort.get_available_providers()}
print(json.dumps(VERSIONS, indent=1))
if GPU is None or 'T4' not in GPU: print('NOTE: not a T4 GPU; the latency numbers below are for', GPU or 'CPU')

REPORT = []
def check(name, expected, actual, ok=None):
    """hard assert; every check is logged as expected / actual / pass"""
    ok = (expected == actual) if ok is None else bool(ok)
    j = lambda v: v if isinstance(v, (str, int, float, bool, type(None), list, dict)) else str(v)
    REPORT.append({'check': name, 'expected': j(expected), 'actual': j(actual), 'pass': ok})
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected {expected}, actual {actual}")
    assert ok, f'check failed: {name} (expected {expected}, actual {actual})'
def info(name, value):
    REPORT.append({'check': name, 'expected': None, 'actual': value, 'pass': True, 'status': 'info'}); print(f'INFO  {name}: {value}')
def skip(name, why):
    REPORT.append({'check': name, 'expected': None, 'actual': why, 'pass': True, 'status': 'skipped'}); print(f'SKIPPED  {name}: {why}')
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
def rel(p): return Path(p).relative_to(ROOT).as_posix()
print('all', len(P), 'inputs found; device', DEVICE, GPU or '')

## 2. Checkpoint (weights_only=True) and model

In [ ]:
CKPT_SHA = sha256(P['ckpt'])
ck = torch.load(P['ckpt'], map_location='cpu', weights_only=True)
check('checkpoint has model, classes, img_size', [], sorted({'model', 'classes', 'img_size'} - set(ck)))
check('checkpoint classes == CLASSES', CLASSES, list(ck['classes']))
check('checkpoint img_size', EXPECT['img_size'], int(ck['img_size']))
IMG = int(ck['img_size'])
for k, v in (('seed', 43), ('lr', 0.002)):           # the file name says s43 / lr0.002; the stored values, if any, must agree
    if k in ck: check(f'checkpoint {k}', v, ck[k])
    else: print(f'checkpoint stores no {k!r}')
print('logit_bias stored:', 'logit_bias' in ck, '-> NOT applied (raw logits only)')

Model class, copied verbatim from `notebooks/track1/train_cls_2026-10-25.ipynb`:

In [ ]:
class Pretrained(nn.Module):
    def __init__(self, n_classes, weights, mode, dropout):
        super().__init__()
        net = torchvision.models.resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(net.fc.in_features, n_classes))
        self.mode = mode
        if mode == 'frozen':
            for p in self.backbone.parameters(): p.requires_grad = False
    def train(self, on=True):
        super().train(on)
        if self.mode == 'frozen': self.backbone.eval()       # BatchNorm statistics stay fixed
        return self
    def forward(self, x): return self.head(self.backbone(x))

In [ ]:
model = Pretrained(8, weights=None, mode='full', dropout=0.2)
model.load_state_dict(ck['model'], strict=True)
model = model.eval().to(DEVICE)
STATE_KEYS = len(ck['model']); del ck
print(f'loaded {STATE_KEYS} tensors (strict); {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters; eval mode')

## 3. Preprocessing and crop recipe (verbatim)

`MEAN`/`STD` and `gpu_eval` from `train_cls_2026-10-25.ipynb` (the class `ac` only carries the `MEAN, STD` line of `aerial_crops.py` from the same notebook, so the `MEAN = ...` line can be copied unchanged). `square()` from `alhada_eval_v2_cell_2026-10-26.py`.

In [ ]:
class ac:   # stands in for aerial_crops.py (train_cls_2026-10-25.ipynb); the next line is copied from it
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, used for scratch and pretrained alike
MEAN = torch.tensor(ac.MEAN).view(1, 3, 1, 1); STD = torch.tensor(ac.STD).view(1, 3, 1, 1)

def gpu_eval(x, out):
    x = x.permute(0, 3, 1, 2).float() / 255
    if x.shape[-1] != out: x = F.interpolate(x, size=(out, out), mode='bilinear', antialias=True, align_corners=False)
    return (x - MEAN.to(x.device)) / STD.to(x.device)

def square(img, x, y, w, h, pad=0.15, out=96):
    cx, cy, s = x + w/2, y + h/2, max(w, h) * (1 + 2*pad)
    return img.crop((round(cx-s/2), round(cy-s/2), round(cx+s/2), round(cy+s/2))).resize((out, out), Image.BICUBIC)

## 4. PyTorch validation reproduction (fp32, no AMP)

Validation rows of CropIndex, read in chunks so that no test row is kept; ordered by `row` as in `train_cls` (`IDX['val']`). Macro-F1 as in `train_cls` `metrics()` (`f1_score(..., labels=LABELS, average='macro', zero_division=0)`). Training logged W&B values under AMP; fp32 here must agree within 0.002.

In [ ]:
VAL = pd.concat([ch[ch.split == 'val'] for ch in
                 pd.read_csv(P['crop_index'], keep_default_na=False, dtype={'ann_id': str}, chunksize=20_000)])
VAL = VAL.sort_values('row').reset_index(drop=True)                  # train_cls: IDX['val']
check('val rows in CropIndex', EXPECT['val_crops'], len(VAL))
check('val row == 0..n-1', True, bool((VAL.row.to_numpy() == np.arange(len(VAL))).all()))
XV = np.load(P['val_npy'])
check('CropsVal shape', (EXPECT['val_crops'], EXPECT['crop_side'], EXPECT['crop_side'], 3), tuple(XV.shape))
check('CropsVal dtype', 'uint8', str(XV.dtype))
K = len(CLASSES); LABELS = list(range(K))
YV = VAL.category.map({c: i for i, c in enumerate(CLASSES)})
check('every val category is a class', 0, int(YV.isna().sum())); YV = YV.to_numpy().astype(int)

### 4b. `square()` reproduces the training crops (Raw frames, read inside the zips)

CropIndex stores the box but not the frame path. The frames are read the way Step 4 (`step4_crops_audit_loader.ipynb`) found them: every zip in `Raw/SeaDronesSee` and `Raw/UAVDT/UAV-benchmark-M*.zip` (not MOTD), zips inside zips included; SeaDronesSee by file name, UAVDT by `<seq>/img<frame>.jpg` under a folder named `M####`. Nothing is extracted to disk: inner zips are opened as file objects inside the outer zip and only the sampled frames are read. Sample: 100 SeaDronesSee + 100 UAVDT **val** rows (seed 42). A frame that is not found, or matches two different members, stops the notebook.

Criterion: max abs pixel difference 0 = PASS; max ≤ 2 and mean < 0.1 = PASS with a note (JPEG decoder difference; Pillow version recorded); otherwise FAIL, the 5 worst crops are printed with their box and the notebook stops.

In [ ]:
import zipfile, PIL
CROP_CHECK_N, CROP_CHECK_SEED = 100, 42
RAW_ZIPS = {'SeaDronesSee': sorted((ROOT / '01_Data/Raw/SeaDronesSee').glob('*.zip')),                     # Step 4: every zip there
            'UAVDT': [z for z in sorted((ROOT / '01_Data/Raw/UAVDT').glob('UAV-benchmark-M*.zip')) if 'MOTD' not in z.name]}
for d, zs in RAW_ZIPS.items():
    check(f'Raw zips for {d}', True, len(zs) > 0)
    check(f'Raw zip paths for {d} contain no Test / AlHada', [], [rel(z) for z in zs if any(f in str(z) for f in FORBIDDEN)])
print({d: [rel(z) for z in zs] for d, zs in RAW_ZIPS.items()})

SAMPLE = pd.concat([VAL[VAL.dataset == d].sample(CROP_CHECK_N, random_state=CROP_CHECK_SEED) for d in ('SeaDronesSee', 'UAVDT')])
check('crop check: sample is val rows only', ['val'], sorted(SAMPLE.split.unique()))
check('crop check: sample size (100 SDS + 100 UAVDT)', {'SeaDronesSee': CROP_CHECK_N, 'UAVDT': CROP_CHECK_N}, SAMPLE.dataset.value_counts().to_dict())

IMG_EXT = {'.jpg', '.jpeg', '.png'}
OPEN_ZIPS = {}                                   # chain of zip names -> ZipFile; inner zips are file objects, nothing written to disk
def walk(zf, chain):
    OPEN_ZIPS[chain] = zf
    for zi in zf.infolist():
        if zi.is_dir(): continue
        if zi.filename.lower().endswith('.zip'): yield from walk(zipfile.ZipFile(zf.open(zi)), chain + (zi.filename,))
        else: yield chain, zi
LOC = {'SeaDronesSee': {}, 'UAVDT': {}}           # Step 4 lookup key -> [(chain, ZipInfo)]
t0 = time.time()
for d, zs in RAW_ZIPS.items():
    for z in zs:
        for chain, zi in walk(zipfile.ZipFile(z), (rel(z),)):
            p = Path(zi.filename)
            if d == 'SeaDronesSee' and p.suffix.lower() in IMG_EXT: LOC[d].setdefault(p.name, []).append((chain, zi))      # sds_files[name]
            if d == 'UAVDT' and re.fullmatch(r'M\d{4}', p.parent.name): LOC[d].setdefault(f'{p.parent.name}/{p.name}', []).append((chain, zi))   # seq_dirs[seq] / img
print(f"zip index: {len(LOC['SeaDronesSee'])} SeaDronesSee images, {len(LOC['UAVDT'])} UAVDT frames, {len(OPEN_ZIPS)} zips ({time.time() - t0:.0f}s)")

SAMPLE['key'] = [Path(f).name if d == 'SeaDronesSee' else f for d, f in zip(SAMPLE.dataset, SAMPLE.image)]   # Step 4 lookup
def distinct(h): return list({(zi.CRC, zi.file_size): (c, zi) for c, zi in h}.values())                      # identical copies count once
hits = [distinct(LOC[d].get(k, [])) for d, k in zip(SAMPLE.dataset, SAMPLE.key)]
miss = SAMPLE[[len(h) == 0 for h in hits]]; amb = SAMPLE[[len(h) > 1 for h in hits]]
if len(miss): print('frames not found:'); print(miss[['crop_id', 'dataset', 'image']].to_string(index=False))
if len(amb): print('frames matching two different members:'); print(amb[['crop_id', 'dataset', 'image']].to_string(index=False))
check('crop check: every sampled frame found in the Raw zips', 0, len(miss))
check('crop check: no sampled frame matches two different zip members', 0, len(amb))
SAMPLE['zloc'] = [h[0] for h in hits]
member_paths = sorted({'/'.join(c) + '/' + zi.filename for c, zi in SAMPLE.zloc})
check('crop check: no member read contains Test / AlHada', [], [m for m in member_paths if any(f in m for f in FORBIDDEN)])

need = {(c, zi.filename): (c, zi) for c, zi in SAMPLE.zloc}
FRAMES = {}
for c, zi in sorted(need.values(), key=lambda t: (t[0], t[1].header_offset)):          # forward order inside each zip
    FRAMES[(c, zi.filename)] = OPEN_ZIPS[c].read(zi)
check('crop check: members read == distinct frames of the 200 val rows', len(need), len(FRAMES))

res = []
for r in SAMPLE.itertuples():
    with Image.open(io.BytesIO(FRAMES[(r.zloc[0], r.zloc[1].filename)])) as im:
        im = im.convert('RGB')                                                           # as Step 4
        c = np.asarray(square(im, r.x, r.y, r.w, r.h))
    dd = np.abs(c.astype(np.int16) - XV[r.row].astype(np.int16))
    res.append({'crop_id': r.crop_id, 'row': r.row, 'dataset': r.dataset, 'image': r.image, 'box_xywh': [r.x, r.y, r.w, r.h],
                'max': int(dd.max()), 'mean': float(dd.mean())})
CC_RES = pd.DataFrame(res); del FRAMES
for z in OPEN_ZIPS.values(): z.close()
OPEN_ZIPS.clear()
CC_MAX, CC_MEAN = int(CC_RES['max'].max()), float(CC_RES['mean'].mean())
CC_STATUS = 'exact' if CC_MAX == 0 else 'pass with note: JPEG decoder difference' if (CC_MAX <= 2 and CC_MEAN < 0.1) else 'FAIL'
print(f'crop check: {len(CC_RES)} crops, max abs pixel diff {CC_MAX}, mean {CC_MEAN:.4f}, Pillow {PIL.__version__} -> {CC_STATUS}')
print(CC_RES.groupby('dataset')[['max', 'mean']].agg(['max', 'mean']).to_string())
if CC_STATUS == 'FAIL':
    print('5 worst crops:'); print(CC_RES.sort_values(['max', 'mean'], ascending=False).head(5).to_string(index=False))
elif CC_STATUS != 'exact': print('NOTE: small differences only; recorded with the Pillow version', PIL.__version__)
CROP_CHECK = {'n': len(CC_RES), 'per_dataset': SAMPLE.dataset.value_counts().to_dict(), 'seed': CROP_CHECK_SEED,
              'max_abs_pixel_diff': CC_MAX, 'mean_abs_pixel_diff': CC_MEAN, 'status': CC_STATUS, 'pillow': PIL.__version__,
              'raw_zips': {d: [rel(z) for z in zs] for d, zs in RAW_ZIPS.items()}}
check('square() reproduces the training crops (max 0; or max <= 2 and mean < 0.1)', 'max 0, or max <= 2 and mean < 0.1',
      f'max {CC_MAX}, mean {CC_MEAN:.4f} ({CC_STATUS})', ok=CC_STATUS != 'FAIL')

### 4c. PyTorch fp32 on all val crops

In [ ]:
XIN = np.empty((len(XV), 3, IMG, IMG), np.float32)                   # the model input, reused for the ONNX parity
LT = np.empty((len(XV), K), np.float32)
with torch.no_grad():
    for i in range(0, len(XV), BATCH):
        x = gpu_eval(torch.from_numpy(XV[i:i + BATCH]).to(DEVICE), IMG)
        XIN[i:i + BATCH] = x.cpu().numpy(); LT[i:i + BATCH] = model(x).float().cpu().numpy()
PRED_T = LT.argmax(1)
F1_T = float(f1_score(YV, PRED_T, labels=LABELS, average='macro', zero_division=0)); ACC_T = float((PRED_T == YV).mean())
print(f'PyTorch fp32 val: macro-F1 {F1_T:.6f} (W&B {EXPECT["val_macro_f1"]}), accuracy {ACC_T:.4f} (W&B {EXPECT["val_acc_wandb"]})')
check(f'val macro-F1 within {EXPECT["val_macro_f1_tol"]} of W&B {EXPECT["val_macro_f1"]} (else preprocessing or checkpoint is wrong)',
      f'|diff| <= {EXPECT["val_macro_f1_tol"]}', f'{F1_T:.6f} (diff {F1_T - EXPECT["val_macro_f1"]:+.6f})',
      ok=abs(F1_T - EXPECT['val_macro_f1']) <= EXPECT['val_macro_f1_tol'])
info('val accuracy (PyTorch fp32; W&B 0.9453)', round(ACC_T, 6))

## 5. Output folder (never overwritten) and ONNX export

In [ ]:
TODAY = time.strftime('%Y-%m-%d')
EXP_DIR = ROOT / '02_Classification/Export'; EXP_DIR.mkdir(parents=True, exist_ok=True)
_done = [int(m.group(1)) for f in os.listdir(EXP_DIR) if (m := re.fullmatch(rf'{TODAY}_ClsResNet18_s43_v(\d+)', f))]   # _INCOMPLETE does not count
VERSION = max(_done, default=0) + 1
FINAL = EXP_DIR / f'{TODAY}_ClsResNet18_s43_v{VERSION}'
OUT = EXP_DIR / (FINAL.name + '_INCOMPLETE')
if OUT.exists():                                   # never deleted automatically
    print(f'a failed run left {OUT.name}: inspect and delete it manually, then re-run')
    raise FileExistsError(f'{rel(OUT)} exists')
assert not FINAL.exists()
OUT.mkdir(); OUT_REL = rel(FINAL)
print('output folder:', OUT_REL, '(written as', OUT.name, 'until all checks pass)')

ONNX_PATH = OUT / 'model.onnx'
model_cpu = model.to('cpu').eval()
dummy = torch.from_numpy(XIN[:2].copy())
kw = {'dynamo': False} if 'dynamo' in inspect.signature(torch.onnx.export).parameters else {}   # classic exporter: dynamic_axes
torch.onnx.export(model_cpu, dummy, str(ONNX_PATH), opset_version=OPSET, input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}}, do_constant_folding=True, **kw)
model = model_cpu.to(DEVICE)

m = onnx.load(str(ONNX_PATH))
for k, v in {'classes': json.dumps(CLASSES), 'input': 'float32 [N,3,64,64], RGB, /255, ImageNet mean/std (see classify.py)',
             'output': 'raw logits [N,8], CLASSES order, no logit bias', 'checkpoint': IN['ckpt'], 'checkpoint_sha256': CKPT_SHA}.items():
    m.metadata_props.append(onnx.StringStringEntryProto(key=k, value=v))
onnx.save(m, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH)); check('onnx.checker.check_model', 'pass', 'pass')
m = onnx.load(str(ONNX_PATH))
dims = lambda t: [d.dim_param or d.dim_value for d in t.type.tensor_type.shape.dim]
check('opset', OPSET, max(o.version for o in m.opset_import if o.domain in ('', 'ai.onnx')))
check('ONNX input', [('input', 1, ['N', 3, IMG, IMG])], [(t.name, t.type.tensor_type.elem_type, dims(t)) for t in m.graph.input])
check('ONNX output', [('logits', 1, ['N', K])], [(t.name, t.type.tensor_type.elem_type, dims(t)) for t in m.graph.output])
ONNX_SHA = sha256(ONNX_PATH); del m
print(f'model.onnx: {ONNX_PATH.stat().st_size / 1e6:.1f} MB, sha256 {ONNX_SHA[:16]}...')

## 6. Parity on all 13,800 val crops: PyTorch fp32 vs ONNX Runtime

Same model inputs (`XIN`) for both. CPUExecutionProvider always; CUDAExecutionProvider only if this onnxruntime build has it **and** the session really uses it (ONNX Runtime falls back to CPU silently otherwise; that case is reported as not available).

In [ ]:
def ort_logits(sess, X):
    return np.concatenate([sess.run(['logits'], {'input': X[i:i + BATCH]})[0] for i in range(0, len(X), BATCH)])

def top2_margin(L): s = np.sort(L.astype(np.float64), 1); return s[:, -1] - s[:, -2]
MARGIN_T = top2_margin(LT)                     # PyTorch top-1 minus top-2 logit, per crop
PARITY, SESS = {}, {}
for prov in ['CPUExecutionProvider', 'CUDAExecutionProvider']:
    name = prov.replace('ExecutionProvider', '')
    if prov not in ort.get_available_providers():
        PARITY[name] = {'available': False, 'reason': 'not in this onnxruntime build'}; skip(f'parity {name}', 'provider not available'); continue
    sess = ort.InferenceSession(str(ONNX_PATH), providers=[prov])
    if sess.get_providers()[0] != prov:
        PARITY[name] = {'available': False, 'reason': f'session fell back to {sess.get_providers()}'}
        skip(f'parity {name}', f'provider could not be initialised, session uses {sess.get_providers()}'); continue
    SESS[name] = sess; t0 = time.time()
    LO = ort_logits(sess, XIN); d = np.abs(LO.astype(np.float64) - LT.astype(np.float64))
    pred = LO.argmax(1); f1 = float(f1_score(YV, pred, labels=LABELS, average='macro', zero_division=0))
    flip = pred != PRED_T; n_flip = int(flip.sum()); n_flip_bad = int((flip & (MARGIN_T >= 2 * PARITY_ATOL)).sum())
    PARITY[name] = {'available': True, 'max_abs_logit_diff': float(d.max()), 'mean_abs_logit_diff': float(d.mean()),
                    'argmax_agreement': float((~flip).mean()), 'argmax_flips': n_flip, 'argmax_flips_near_tie': n_flip - n_flip_bad,
                    'val_macro_f1': f1, 'val_accuracy': float((pred == YV).mean()), 'seconds': round(time.time() - t0, 1)}
    print(f'{name}: max |diff| {d.max():.3e}, mean {d.mean():.3e}, argmax flips {n_flip} (near-tie {n_flip - n_flip_bad}), macro-F1 {f1:.6f}')
    check(f'{name}: max abs logit diff <= {PARITY_ATOL}', f'<= {PARITY_ATOL}', f'{d.max():.3e}', ok=d.max() <= PARITY_ATOL)
    check(f'{name}: argmax flips only where PyTorch top-2 margin < {2 * PARITY_ATOL}', 0, n_flip_bad)
    info(f'{name}: argmax flips at near-ties (top-2 margin < {2 * PARITY_ATOL})', n_flip)
    check(f'{name}: |ONNX val macro-F1 - PyTorch val macro-F1| <= 1e-4', '<= 1e-4', f'{abs(f1 - F1_T):.2e}', ok=abs(f1 - F1_T) <= 1e-4)
check('CPU provider parity ran', True, PARITY['CPU']['available'])
if not PARITY['CUDA']['available']: print('CUDAExecutionProvider NOT available:', PARITY['CUDA']['reason'])

## 7. `classify.py` for integration, consistency checks, latency

`classify.py` is written into the output folder and imported from there. Its `square()` and `gpu_eval` are the same verbatim copies as above; the checks below compare it with this notebook on crops, preprocessing and logits.

Latency uses a **synthetic** 1920×1080 frame (random pixels, seed 42) and random boxes of 15–80 px: no test frame and no Al-Hada frame may be opened here, and the timing does not depend on the pixel content. In `classify()` the crop and resize run on the CPU (PIL, torch), the model on the stated provider.

In [ ]:
CLASSIFY_SRC = r'''"""Track 1 aerial object classifier (ONNX Runtime), for integration.

Model: ResNet18, full fine-tune, lr 0.002, 64 px input, seed 43 (selected on validation macro-F1 only).
Classes, index order of the logits (CLASSES below):
    0 swimmer, 1 boat, 2 jetski, 3 life_saving_appliances, 4 buoy, 5 car, 6 truck, 7 bus

Input contract
- frame_rgb: numpy uint8 array, H x W x 3, RGB channel order. NOT BGR: convert OpenCV frames first (frame[..., ::-1]).
- boxes_xywh: array-like [K, 4] in frame pixels: x, y = top-left corner, w, h = width, height. w and h must be > 0,
  otherwise ValueError.
- Each box is cut as in Track 1 Phase 1: a square centred on the box, side = max(w, h) * (1 + 2 * 0.15), PIL crop
  (areas outside the frame are filled by PIL with black), resized to 96 x 96 BICUBIC; then float / 255, resized to
  64 x 64 (bilinear, antialias, torch), normalised with the ImageNet mean / std.
- Output: RAW logits [K, 8]. No logit bias is applied. confidence = max softmax of the raw logits.

Functions
- load(onnx_path, providers=None) -> onnxruntime.InferenceSession (CUDA first if available, else CPU)
- classify(session, frame_rgb, boxes_xywh) -> list of (class_name, confidence), one per box
- classify_probs(session, frame_rgb, boxes_xywh) -> numpy float32 [K, 8] softmax probabilities (CLASSES order)
- classify_logits(session, frame_rgb, boxes_xywh) -> numpy float32 [K, 8] raw logits
- crops(frame_rgb, boxes_xywh) -> numpy uint8 [K, 96, 96, 3]
Dependencies: numpy, Pillow, torch (resize only), onnxruntime (or onnxruntime-gpu).
"""
import numpy as np, torch, torch.nn.functional as F, onnxruntime as ort
from PIL import Image

CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']
IMG_SIZE, CROP_SIZE = 64, 96

class ac:   # stands in for aerial_crops.py of train_cls_2026-10-25.ipynb; the next line is copied from it
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, used for scratch and pretrained alike
MEAN = torch.tensor(ac.MEAN).view(1, 3, 1, 1); STD = torch.tensor(ac.STD).view(1, 3, 1, 1)

# crop recipe, copied verbatim from alhada_eval_v2_cell_2026-10-26.py (Phase 1)
def square(img, x, y, w, h, pad=0.15, out=96):
    cx, cy, s = x + w/2, y + h/2, max(w, h) * (1 + 2*pad)
    return img.crop((round(cx-s/2), round(cy-s/2), round(cx+s/2), round(cy+s/2))).resize((out, out), Image.BICUBIC)

# eval preprocessing, copied verbatim from train_cls_2026-10-25.ipynb (runs on the tensor's device; CPU here)
def gpu_eval(x, out):
    x = x.permute(0, 3, 1, 2).float() / 255
    if x.shape[-1] != out: x = F.interpolate(x, size=(out, out), mode='bilinear', antialias=True, align_corners=False)
    return (x - MEAN.to(x.device)) / STD.to(x.device)

def load(onnx_path, providers=None):
    """ONNX Runtime session. providers=None: CUDA if this onnxruntime build has it, then CPU."""
    if providers is None:
        providers = [p for p in ('CUDAExecutionProvider', 'CPUExecutionProvider') if p in ort.get_available_providers()]
    return ort.InferenceSession(str(onnx_path), providers=providers)


def _check(frame_rgb, boxes_xywh):
    if not (isinstance(frame_rgb, np.ndarray) and frame_rgb.dtype == np.uint8 and frame_rgb.ndim == 3 and frame_rgb.shape[2] == 3):
        raise ValueError('frame_rgb must be a numpy uint8 array H x W x 3 (RGB)')
    b = np.asarray(boxes_xywh, dtype=np.float64)
    if b.size == 0: return b.reshape(0, 4)
    if b.ndim != 2 or b.shape[1] != 4: raise ValueError(f'boxes_xywh must be [K, 4] (x, y, w, h), got shape {b.shape}')
    if not np.isfinite(b).all(): raise ValueError('boxes_xywh contains NaN or inf')
    bad = np.flatnonzero((b[:, 2] <= 0) | (b[:, 3] <= 0))
    if len(bad): raise ValueError(f'boxes with w or h <= 0 at rows {bad.tolist()}')
    return b


def crops(frame_rgb, boxes_xywh):
    """uint8 [K, 96, 96, 3] square crops, one per box."""
    b = _check(frame_rgb, boxes_xywh)
    if len(b) == 0: return np.zeros((0, CROP_SIZE, CROP_SIZE, 3), np.uint8)
    img = Image.fromarray(frame_rgb)
    return np.stack([np.asarray(square(img, *map(float, r))) for r in b])


def classify_logits(session, frame_rgb, boxes_xywh):
    """raw logits, float32 [K, 8], CLASSES order (no logit bias)."""
    X = crops(frame_rgb, boxes_xywh)
    if len(X) == 0: return np.zeros((0, len(CLASSES)), np.float32)
    with torch.no_grad(): x = gpu_eval(torch.from_numpy(X), IMG_SIZE).numpy().astype(np.float32)
    return session.run(['logits'], {'input': x})[0]


def classify_probs(session, frame_rgb, boxes_xywh):
    """softmax of the raw logits, float32 [K, 8], CLASSES order."""
    L = classify_logits(session, frame_rgb, boxes_xywh).astype(np.float64)
    e = np.exp(L - L.max(1, keepdims=True)); return (e / e.sum(1, keepdims=True)).astype(np.float32)


def classify(session, frame_rgb, boxes_xywh):
    """list of (class_name, confidence) per box; confidence = max softmax of the raw logits."""
    P = classify_probs(session, frame_rgb, boxes_xywh)
    return [(CLASSES[int(k)], float(p[k])) for p, k in zip(P, P.argmax(1))]
'''

In [ ]:
(OUT / 'classify.py').write_text(CLASSIFY_SRC)
os.sys.dont_write_bytecode = True                  # no __pycache__ in the output folder
spec = importlib.util.spec_from_file_location('classify', OUT / 'classify.py'); clf = importlib.util.module_from_spec(spec); spec.loader.exec_module(clf)
check('classify.CLASSES == CLASSES', CLASSES, clf.CLASSES)
check('classify.IMG_SIZE == checkpoint img_size', IMG, clf.IMG_SIZE)

rng = np.random.default_rng(E2E_SEED)
FH, FW = E2E_FRAME_HW
FRAME = rng.integers(0, 256, (FH, FW, 3), dtype=np.uint8)
def rand_boxes(n):
    w, h = rng.uniform(*E2E_SIDE_PX, n), rng.uniform(*E2E_SIDE_PX, n)
    return np.stack([rng.uniform(0, FW - w), rng.uniform(0, FH - h), w, h], 1)
EDGE = np.array([[-10.0, -5.0, 30.0, 20.0], [FW - 12.0, FH - 9.0, 25.0, 40.0], [3.5, 700.2, 61.7, 18.3]])   # partly outside / fractional
BOXES = {n: rand_boxes(n) for n in E2E_BOXES}
B_ALL = np.concatenate([BOXES[100], EDGE])

im = Image.fromarray(FRAME)
X_nb = np.stack([np.asarray(square(im, *map(float, b))) for b in B_ALL])
check('classify.crops == notebook square() (byte-identical, incl. boxes partly outside the frame)', True, bool(np.array_equal(clf.crops(FRAME, B_ALL), X_nb)))
with torch.no_grad():
    same = torch.equal(clf.gpu_eval(torch.from_numpy(XV[:BATCH]), IMG), gpu_eval(torch.from_numpy(XV[:BATCH]), IMG))
check('classify.gpu_eval == notebook gpu_eval on 512 val crops (CPU, exact)', True, same)

E2E_PROV = 'CUDA' if 'CUDA' in SESS else 'CPU'; sess = SESS[E2E_PROV]
L_clf = clf.classify_logits(sess, FRAME, B_ALL)
with torch.no_grad(): L_ref = model(gpu_eval(torch.from_numpy(X_nb).to(DEVICE), IMG)).float().cpu().numpy()
d = float(np.abs(L_clf.astype(np.float64) - L_ref).max())
check(f'classify_logits ({E2E_PROV}) vs PyTorch on the same crops: max abs diff <= {PARITY_ATOL}', f'<= {PARITY_ATOL}', f'{d:.3e}', ok=d <= PARITY_ATOL)
flip = L_clf.argmax(1) != L_ref.argmax(1)
check(f'classify_logits argmax flips only where PyTorch top-2 margin < {2 * PARITY_ATOL}', 0, int((flip & (top2_margin(L_ref) >= 2 * PARITY_ATOL)).sum()))
info('classify_logits argmax flips at near-ties', int(flip.sum()))
Pp = clf.classify_probs(sess, FRAME, B_ALL); res = clf.classify(sess, FRAME, B_ALL)
check('classify_probs rows sum to 1', True, bool(np.allclose(Pp.sum(1), 1, atol=1e-5)))
check('classify() = (CLASSES[argmax], max softmax of raw logits)', True,
      all(c == CLASSES[k] and abs(p - Pp[i, k]) < 1e-7 for i, ((c, p), k) in enumerate(zip(res, Pp.argmax(1)))))
for bad in ([[10, 10, 0, 20]], [[10, 10, 20, -1]]):
    try: clf.classify(sess, FRAME, bad); raised = False
    except ValueError: raised = True
    check(f'classify raises ValueError for box {bad[0]}', True, raised)
check('classify of 0 boxes -> empty list', [], clf.classify(sess, FRAME, np.zeros((0, 4))))

In [ ]:
def timeit(fn):
    for _ in range(LAT_WARMUP): fn()
    t = []
    for _ in range(LAT_RUNS):
        t0 = time.perf_counter(); fn(); t.append((time.perf_counter() - t0) * 1e3)
    t = np.array(t); return {'p50_ms': float(np.percentile(t, 50)), 'p95_ms': float(np.percentile(t, 95)), 'p99_ms': float(np.percentile(t, 99)), 'max_ms': float(t.max())}

LAT = {'provider': E2E_PROV, 'device': GPU or 'CPU', 'warmup': LAT_WARMUP, 'runs': LAT_RUNS, 'rows': []}
if E2E_PROV != 'CUDA': print('latency measured with the CPU provider: CUDAExecutionProvider not available')
for bs in LAT_BATCHES:
    x = np.ascontiguousarray(XIN[:bs])
    LAT['rows'].append({'what': f'model only, batch {bs}', **timeit(lambda: sess.run(['logits'], {'input': x}))})
for n in E2E_BOXES:
    b = BOXES[n]
    LAT['rows'].append({'what': f'classify(), 1920x1080 frame, {n} boxes', **timeit(lambda: clf.classify(sess, FRAME, b))})
LAT_TABLE = pd.DataFrame(LAT['rows']).set_index('what').round(2)
print(f"latency, ONNX Runtime {E2E_PROV} provider on {LAT['device']} ({LAT_WARMUP} warm-up + {LAT_RUNS} timed runs):")
print(LAT_TABLE.to_string())
info('latency (ms), info only: the 200 ms p95 criterion is for the whole system', LAT['rows'])

## 8. README, report, hygiene, rename

In [ ]:
P_ROWS = [(k, v) for k, v in PARITY.items()]
par_md = '\n'.join(f"| {k} | {v['max_abs_logit_diff']:.2e} | {v['mean_abs_logit_diff']:.2e} | {v['argmax_agreement'] * 100:.2f}% | {v['argmax_flips']} | {v['val_macro_f1']:.4f} |"
                   if v['available'] else f"| {k} | not available ({v['reason']}) | | | | |" for k, v in P_ROWS)
lat_md = '\n'.join(f"| {r['what']} | {r['p50_ms']:.2f} | {r['p95_ms']:.2f} | {r['p99_ms']:.2f} | {r['max_ms']:.2f} |" for r in LAT['rows'])
README = f"""# Track 1 classifier, ONNX export ({TODAY}, v{VERSION})

Files: `model.onnx`, `classify.py`, `export_report.json`, this README. All paths are relative to the project Drive folder.

## Model and selection
- Checkpoint: {IN['ckpt']} (sha256 {CKPT_SHA})
- Selection rule (fixed before export): {SELECTION_RULE}
- **Raw logits, no logit bias.** The checkpoint's per-class logit bias is not applied (dataset card: raw scores are the headline; the bias hurt calibration).
- model.onnx: opset {OPSET}, input `input` float32 [N, 3, {IMG}, {IMG}], output `logits` float32 [N, {K}], dynamic batch. sha256 {ONNX_SHA}.

## Input contract (classify.py)
- `load(onnx_path, providers=None)` -> session (CUDA first if available, else CPU).
- `classify(session, frame_rgb, boxes_xywh)` -> list of (class_name, confidence); `classify_probs(...)` -> [K, 8] probabilities; `classify_logits(...)` -> [K, 8] raw logits.
- `frame_rgb`: numpy uint8 H x W x 3, **RGB** (not BGR). `boxes_xywh`: [K, 4], x, y (top-left), w, h in frame pixels; w or h <= 0 raises ValueError.
- Per box: square crop centred on the box, side max(w, h) x 1.3 (pad 0.15), PIL crop (outside the frame filled black), 96 x 96 BICUBIC -> /255 -> 64 x 64 bilinear antialias -> ImageNet mean/std. Same code as training and Phase 1.
- Class order: {', '.join(f'{i} {c}' for i, c in enumerate(CLASSES))}.
- confidence = max softmax of the raw logits.

## Validation results ({len(YV):,} val crops; test and Al-Hada not used)
PyTorch fp32 val macro-F1 {F1_T:.4f} (W&B {EXPECT['val_macro_f1']}), accuracy {ACC_T:.4f}.

Crop recipe check: {CROP_CHECK['n']} val crops (100 SeaDronesSee + 100 UAVDT, seed {CROP_CHECK['seed']}) re-cut with square() from the Raw frames match the training crops: max abs pixel diff {CROP_CHECK['max_abs_pixel_diff']}, mean {CROP_CHECK['mean_abs_pixel_diff']:.4f} ({CROP_CHECK['status']}; Pillow {CROP_CHECK['pillow']}).

| ONNX Runtime provider | max abs logit diff | mean abs diff | argmax agreement | argmax flips (near-tie) | val macro-F1 |
|---|---|---|---|---|---|
{par_md}

Acceptance: max abs logit diff <= {PARITY_ATOL}; an argmax difference only where the PyTorch top-2 logit margin is < {2 * PARITY_ATOL} (counted in the table); |ONNX - PyTorch val macro-F1| <= 1e-4.

## Latency ({LAT['provider']} provider, {LAT['device']}, {LAT_WARMUP} warm-up + {LAT_RUNS} timed runs)
| what | p50 ms | p95 ms | p99 ms | max ms |
|---|---|---|---|---|
{lat_md}

classify(): crop and resize on the CPU, model on the {LAT['provider']} provider; synthetic 1920x1080 frame, boxes 15-80 px. Info only: the 200 ms p95 criterion applies to the whole system; these numbers are the classifier's share.

## Known weak points
- truck -> car and bus -> car confusions, worst on small objects.
- Al-Hada (held-out real footage, evaluation only): macro-F1 about 0.33.
"""
REP = {'created': time.strftime('%Y-%m-%d %H:%M'), 'output': OUT_REL,
       'checkpoint': {'path': IN['ckpt'], 'sha256': CKPT_SHA}, 'selection_rule': SELECTION_RULE, 'classes': CLASSES,
       'img_size': IMG, 'raw_logits_no_logit_bias': True,
       'onnx': {'file': 'model.onnx', 'sha256': ONNX_SHA, 'opset': OPSET, 'input': ['input', 'float32', ['N', 3, IMG, IMG]],
                'output': ['logits', 'float32', ['N', K]]},
       'val_reproduction': {'pytorch_fp32_macro_f1': F1_T, 'pytorch_fp32_accuracy': ACC_T, 'wandb_macro_f1': EXPECT['val_macro_f1'],
                            'wandb_accuracy': EXPECT['val_acc_wandb'], 'n_crops': len(YV)},
       'crop_recipe_check': CROP_CHECK, 'parity': PARITY, 'latency': LAT, 'versions': VERSIONS, 'inputs': {k: v for k, v in IN.items()}}
BAD = ['/content/', '/Users/', 'MyDrive']
def report_json():
    return json.dumps({**REP, 'checks': REPORT, 'all_pass': all(r['pass'] for r in REPORT),
                       'skipped': [r['check'] for r in REPORT if r.get('status') == 'skipped']}, indent=1, default=str)
check('no absolute paths in README.md', [], [b for b in BAD if b in README])
check('no absolute paths in classify.py', [], [b for b in BAD if b in CLASSIFY_SRC])
onnx_bytes = ONNX_PATH.read_bytes()
check('no absolute paths in model.onnx', [], [b for b in BAD if b.encode() in onnx_bytes]); del onnx_bytes
check('no absolute paths in export_report.json', [], [b for b in BAD if b in report_json()])   # scanned before writing
RJ = report_json(); assert not any(b in RJ for b in BAD)
(OUT / 'README.md').write_text(README); (OUT / 'export_report.json').write_text(RJ)
check('output files', ['README.md', 'classify.py', 'export_report.json', 'model.onnx'], sorted(p.name for p in OUT.iterdir()))
check('model.onnx unchanged since export', ONNX_SHA, sha256(ONNX_PATH))
assert all(r['pass'] for r in REPORT) and not FINAL.exists()
OUT.rename(FINAL)
print('renamed ->', OUT_REL)

## 9. Summary

In [ ]:
print(f'val macro-F1: PyTorch fp32 {F1_T:.6f} | ' + ' | '.join(f"ONNX {k} {v['val_macro_f1']:.6f}" for k, v in PARITY.items() if v['available']))
print('parity max abs logit diff:', {k: (f"{v['max_abs_logit_diff']:.2e}" if v['available'] else 'not available') for k, v in PARITY.items()})
print(f"\nlatency p95 (ms), {LAT['provider']} provider on {LAT['device']}:"); print(LAT_TABLE[['p95_ms']].to_string())
print(); print(pd.DataFrame([{'check': r['check'], 'result': {'skipped': 'SKIPPED', 'info': 'INFO'}.get(r.get('status'), 'PASS' if r['pass'] else 'FAIL')}
                             for r in REPORT]).to_string(index=False))
n = {s: sum(r.get('status') == s for r in REPORT) for s in ('skipped', 'info')}
print(f"\nALL {len(REPORT) - n['skipped'] - n['info']} CHECKS PASS ({n['skipped']} skipped, {n['info']} info) -> {OUT_REL}")